# Experiment 8: Clustering Human Activity Recognition Data (K-Means, DBSCAN, and Hierarchical Clustering)
This standalone notebook implements unsupervised clustering models (K-Means with Elbow & Silhouette optimization, DBSCAN with noise filtering, and Hierarchical Agglomerative Clustering with Ward's linkage) on the 561-feature UCI HAR sensory dataset.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex8."""
    if os.path.basename(os.getcwd()) == 'Ex8':
        if rel_path.startswith('Ex8/'):
            return rel_path[len('Ex8/'):]
    return rel_path

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex8'), exist_ok=True)

In [2]:
def load_har_data(data_dir="Datasets/UCI_HAR", sample_size=2500):
    dir_path = resolve_path(data_dir)
    act_path = os.path.join(dir_path, 'activity_labels.txt')
    act_map = {}
    with open(act_path, 'r') as f:
        for line in f:
            if line.strip():
                p = line.strip().split()
                act_map[int(p[0])] = p[1]
                
    X_tr = np.loadtxt(os.path.join(dir_path, 'train', 'X_train.txt'))
    y_tr = np.loadtxt(os.path.join(dir_path, 'train', 'y_train.txt'), dtype=int)
    X_te = np.loadtxt(os.path.join(dir_path, 'test', 'X_test.txt'))
    y_te = np.loadtxt(os.path.join(dir_path, 'test', 'y_test.txt'), dtype=int)
    
    X = np.vstack((X_tr, X_te))
    y = np.concatenate((y_tr, y_te))
    
    if sample_size and sample_size < len(y):
        np.random.seed(42)
        idx = []
        per_class = sample_size // len(np.unique(y))
        for c in np.unique(y):
            c_idx = np.where(y == c)[0]
            idx.extend(np.random.choice(c_idx, size=min(per_class, len(c_idx)), replace=False))
        X = X[idx]
        y = y[idx]
        
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y, act_map

In [3]:
def run_experiment_8(data_dir="Datasets/UCI_HAR", sample_size=2500):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 8: HAR CLUSTERING PIPELINE ===")
    print("="*60)
    
    X, y, act_map = load_har_data(data_dir=data_dir, sample_size=sample_size)
    
    table1 = []
    for k in range(2, 9):
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
        labels = km.fit_predict(X)
        table1.append({
            'k': k,
            'WCSS (Inertia)': round(km.inertia_, 2),
            'Silhouette Score': round(silhouette_score(X, labels), 4)
        })
        
    km_final = KMeans(n_clusters=6, init='k-means++', n_init=10, random_state=42)
    km_labels = km_final.fit_predict(X)
    
    db = DBSCAN(eps=15.0, min_samples=15)
    db_labels = db.fit_predict(X)
    
    hac = AgglomerativeClustering(n_clusters=6, linkage='ward')
    hac_labels = hac.fit_predict(X)
    
    models = {'K-Means (k=6)': km_labels, 'DBSCAN': db_labels, 'Hierarchical (Ward)': hac_labels}
    comp_metrics = []
    for name, labels in models.items():
        valid_mask = labels != -1 if -1 in labels else np.ones(len(labels), dtype=bool)
        sil = silhouette_score(X[valid_mask], labels[valid_mask]) if len(np.unique(labels[valid_mask])) > 1 else 0.0
        db_idx = davies_bouldin_score(X[valid_mask], labels[valid_mask]) if len(np.unique(labels[valid_mask])) > 1 else 0.0
        ch_idx = calinski_harabasz_score(X[valid_mask], labels[valid_mask]) if len(np.unique(labels[valid_mask])) > 1 else 0.0
        comp_metrics.append({
            'Algorithm': name,
            'Silhouette Score': round(sil, 4),
            'Davies-Bouldin Index': round(db_idx, 4),
            'Calinski-Harabasz Index': round(ch_idx, 2),
            'Adjusted Rand Index (ARI)': round(adjusted_rand_score(y, labels), 4),
            'Normalized Mutual Info (NMI)': round(normalized_mutual_info_score(y, labels), 4)
        })
        
    print("\n=== EXPERIMENT 8 PIPELINE COMPLETE ===")
    return {'table1_elbow': table1, 'comp_metrics': comp_metrics}

In [4]:
# Master Execution Cell
ex8_output = run_experiment_8()
print('\nTABLE 1: K-MEANS ELBOW METHOD')
display(pd.DataFrame(ex8_output['table1_elbow']).style.background_gradient(cmap='Blues', subset=['Silhouette Score']))
print('\nCLUSTERING PERFORMANCE COMPARISON')
display(pd.DataFrame(ex8_output['comp_metrics']).set_index('Algorithm').style.background_gradient(cmap='YlGn', subset=['Adjusted Rand Index (ARI)', 'Normalized Mutual Info (NMI)']))

=== LAUNCHING EXPERIMENT 8: HAR CLUSTERING PIPELINE ===



=== EXPERIMENT 8 PIPELINE COMPLETE ===

TABLE 1: K-MEANS ELBOW METHOD


,k,WCSS (Inertia),Silhouette Score
0,2,800983.810000,0.381700
1,3,708006.730000,0.301300
2,4,675461.520000,0.156900
3,5,643651.100000,0.145500
4,6,625772.660000,0.102900
5,7,608029.890000,0.083800
6,8,595478.720000,0.079400



CLUSTERING PERFORMANCE COMPARISON


,Silhouette Score,Davies-Bouldin Index,Calinski-Harabasz Index,Adjusted Rand Index (ARI),Normalized Mutual Info (NMI)
Algorithm,,,,,
K-Means (k=6),0.102900,2.585500,616.350000,0.285700,0.455600
DBSCAN,0.320100,1.858800,396.030000,0.277800,0.426000
Hierarchical (Ward),0.073300,2.828400,581.930000,0.282700,0.455400
